# t-SNE (t-distributed Stochastic Neighbor Embedding)

Implementación desde cero del algoritmo t-SNE para reducción de dimensionalidad y visualización de datos de alta dimensión.

---

## ¿Cuál es el objetivo de t-SNE?

Imagina que eres un cartógrafo y te dan la tarea de representar la Tierra (un objeto 3D) en un mapa plano (2D). Necesitas preservar las **relaciones de cercanía**: países vecinos deben quedar juntos en el mapa, aunque inevitablemente habrá distorsiones.

**t-SNE** hace exactamente eso pero con datos. Toma datos que viven en espacios de muchas dimensiones (cientos o miles de variables) y los proyecta a 2D o 3D para que podamos **visualizarlos**, manteniendo la estructura local: **los puntos que eran vecinos en el espacio original siguen siendo vecinos en el mapa**.

### ¿Por qué no usar PCA?

PCA (Análisis de Componentes Principales) también reduce dimensiones, pero tiene una limitación importante: **solo captura relaciones lineales**. Si los datos forman estructuras curvas, anillos o clusters no lineales, PCA no puede representarlas bien.

| Característica | PCA | t-SNE |
|---|---|---|
| Tipo de relaciones | Lineales | No lineales |
| Preserva | Varianza global | Estructura local |
| Determinista | Sí | No (aleatorio) |
| Velocidad | Rápido | Lento |
| Uso principal | Preprocesamiento | Visualización |

### La idea intuitiva

t-SNE funciona en dos pasos fundamentales:

1. **En el espacio original**: mide qué tan "vecinos" son cada par de puntos usando una distribución Gaussiana
2. **En el espacio reducido**: coloca los puntos en 2D y mide sus similitudes usando una distribución t-Student
3. **Optimiza**: mueve los puntos en 2D hasta que las similitudes en ambos espacios coincidan lo mejor posible

---

## Un ejemplo sencillo

Consideremos 6 puntos en 3D que forman dos grupos claros. Queremos proyectarlos a 2D manteniendo la estructura de clusters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(42)

# Dos clusters en 3D
X = np.array([
    [1.0, 1.0, 1.0],   # Cluster A
    [1.2, 0.8, 1.1],   # Cluster A
    [0.9, 1.1, 0.9],   # Cluster A
    [5.0, 5.0, 5.0],   # Cluster B
    [5.2, 4.8, 5.1],   # Cluster B
    [4.9, 5.1, 4.9],   # Cluster B
])

labels = ['A', 'A', 'A', 'B', 'B', 'B']
colors = ['#e74c3c' if l == 'A' else '#3498db' for l in labels]
nombres = [f'$x_{i+1}$' for i in range(6)]

print("Datos en 3D:")
print(f"{'Punto':<8} {'Dim 1':<8} {'Dim 2':<8} {'Dim 3':<8} {'Cluster'}")
print("-" * 42)
for i in range(len(X)):
    print(f"x_{i+1:<5} {X[i,0]:<8.1f} {X[i,1]:<8.1f} {X[i,2]:<8.1f} {labels[i]}")

In [ ]:
fig = plt.figure(figsize=(14, 5))

# Datos originales en 3D
ax1 = fig.add_subplot(131, projection='3d')
for i in range(len(X)):
    ax1.scatter(X[i,0], X[i,1], X[i,2], c=colors[i], s=100, edgecolors='black', zorder=5)
    ax1.text(X[i,0]+0.15, X[i,1]+0.15, X[i,2]+0.15, f'$x_{i+1}$', fontsize=11)
ax1.set_xlabel('Dim 1'); ax1.set_ylabel('Dim 2'); ax1.set_zlabel('Dim 3')
ax1.set_title('Datos originales (3D)', fontsize=13, fontweight='bold')

# PCA manual: proyectar a 2D
X_centered = X - X.mean(axis=0)
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
X_pca = X_centered @ Vt[:2].T

ax2 = fig.add_subplot(132)
for i in range(len(X)):
    ax2.scatter(X_pca[i,0], X_pca[i,1], c=colors[i], s=100, edgecolors='black', zorder=5)
    ax2.annotate(f'$x_{i+1}$', (X_pca[i,0], X_pca[i,1]), textcoords="offset points", xytext=(8,8), fontsize=11)
ax2.set_xlabel('PC 1'); ax2.set_ylabel('PC 2')
ax2.set_title('PCA (2D)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Simulación de lo que haría t-SNE (resultado ideal)
# t-SNE separaría claramente los clusters y preservaría las distancias locales
np.random.seed(10)
X_tsne_sim = np.array([
    [-3.2, 0.5], [-2.8, -0.3], [-3.0, 0.8],
    [3.0, -0.4], [3.3, 0.6], [2.9, -0.7]
])

ax3 = fig.add_subplot(133)
for i in range(len(X)):
    ax3.scatter(X_tsne_sim[i,0], X_tsne_sim[i,1], c=colors[i], s=100, edgecolors='black', zorder=5)
    ax3.annotate(f'$x_{i+1}$', (X_tsne_sim[i,0], X_tsne_sim[i,1]), textcoords="offset points", xytext=(8,8), fontsize=11)
ax3.set_xlabel('t-SNE 1'); ax3.set_ylabel('t-SNE 2')
ax3.set_title('t-SNE (2D) - resultado esperado', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObserva cómo t-SNE agrupa claramente los clusters,")
print("enfatizando la estructura local de los datos.")

Tanto PCA como t-SNE logran separar los clusters en este caso sencillo. Pero cuando los datos tienen **estructuras no lineales complejas**, t-SNE brilla mucho más. La verdadera diferencia se ve con datos de alta dimensionalidad.

Pero... ¿cómo decide t-SNE dónde colocar cada punto en el mapa 2D?

---

## Demostración Matemática

### Paso 1: Similitudes en el espacio original (distribución Gaussiana)

Para cada par de puntos $x_i$ y $x_j$ en el espacio original, definimos la **probabilidad condicional** de que $x_i$ elija a $x_j$ como vecino:

$$p_{j|i} = \frac{\exp\left(-\|x_i - x_j\|^2 / 2\sigma_i^2\right)}{\sum_{k \neq i} \exp\left(-\|x_i - x_k\|^2 / 2\sigma_i^2\right)}$$

Donde:
- $\|x_i - x_j\|^2$ es la distancia euclidiana al cuadrado entre los puntos
- $\sigma_i$ es el ancho de la Gaussiana centrada en $x_i$ (cada punto tiene su propio $\sigma$)
- El denominador normaliza para que las probabilidades sumen 1

**Interpretación**: Si $x_j$ está cerca de $x_i$, entonces $p_{j|i}$ será alta. Si está lejos, será baja.

Para hacer la distribución simétrica (que $p_{ij} = p_{ji}$), usamos:

$$p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}$$

donde $n$ es el número de puntos.

### Paso 2: La perplejidad y el ancho $\sigma_i$

¿Cómo elegimos $\sigma_i$ para cada punto? Usamos la **perplejidad** (perplexity), un hiperparámetro que controla el número efectivo de vecinos:

$$\text{Perp}(P_i) = 2^{H(P_i)}$$

donde $H(P_i)$ es la entropía de Shannon de la distribución condicional:

$$H(P_i) = -\sum_{j \neq i} p_{j|i} \log_2 p_{j|i}$$

Para cada punto $x_i$, buscamos el $\sigma_i$ tal que la perplejidad coincida con el valor deseado (típicamente entre 5 y 50). Esto se hace con **búsqueda binaria**.

- **Perplejidad baja** → $\sigma$ pequeño → solo vecinos muy cercanos importan → clusters más apretados
- **Perplejidad alta** → $\sigma$ grande → más vecinos importan → visión más global

### Paso 3: Similitudes en el espacio reducido (distribución t-Student)

En el espacio de baja dimensión (2D), tenemos puntos $y_i$ y definimos similitudes con una **distribución t-Student con 1 grado de libertad** (equivalente a una Cauchy):

$$q_{ij} = \frac{\left(1 + \|y_i - y_j\|^2\right)^{-1}}{\sum_{k \neq l} \left(1 + \|y_k - y_l\|^2\right)^{-1}}$$

### ¿Por qué t-Student y no Gaussiana en el espacio reducido?

Esta es la innovación clave de t-SNE (y la razón de la "t" en el nombre):

Cuando reducimos dimensiones, ocurre el **problema de hacinamiento** (crowding problem): en alta dimensión hay mucho "espacio" para acomodar vecinos, pero en 2D no. Si usáramos Gaussianas en ambos espacios, los puntos moderadamente distantes se comprimirían encima de los vecinos cercanos.

La distribución t-Student tiene **colas más pesadas** que la Gaussiana. Esto significa que:
- Puntos **cercanos** en alta dimensión → quedan cercanos en 2D (ambas distribuciones coinciden)
- Puntos **lejanos** en alta dimensión → pueden quedar más separados en 2D (la t-Student permite mayores distancias)

### Paso 4: Función de costo — Divergencia KL

Queremos que la distribución $Q$ (espacio reducido) sea lo más parecida posible a $P$ (espacio original). Medimos la diferencia con la **Divergencia de Kullback-Leibler**:

$$C = KL(P \| Q) = \sum_{i} \sum_{j \neq i} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

Propiedades importantes de la KL en t-SNE:
- $KL \geq 0$, y $KL = 0$ solo cuando $P = Q$
- Es **asimétrica**: penaliza más cuando $p_{ij}$ es grande pero $q_{ij}$ es pequeño (puntos cercanos en el original que quedan lejos en 2D)
- Esto significa que t-SNE **prioriza preservar la estructura local**

### Paso 5: Gradiente para optimización

Para minimizar $C$, necesitamos el gradiente con respecto a cada punto $y_i$ del espacio reducido:

$$\frac{\partial C}{\partial y_i} = 4 \sum_{j \neq i} (p_{ij} - q_{ij})(y_i - y_j)\left(1 + \|y_i - y_j\|^2\right)^{-1}$$

**Interpretación del gradiente como fuerzas**:
- Si $p_{ij} > q_{ij}$ → los puntos deberían estar más cerca → **fuerza atractiva**
- Si $p_{ij} < q_{ij}$ → los puntos deberían estar más lejos → **fuerza repulsiva**
- El factor $(1 + \|y_i - y_j\|^2)^{-1}$ modera la fuerza según la distancia actual

### Derivación del gradiente

Partimos de:
$$C = \sum_i \sum_{j \neq i} p_{ij} \log \frac{p_{ij}}{q_{ij}} = \sum_i \sum_{j \neq i} p_{ij} \log p_{ij} - \sum_i \sum_{j \neq i} p_{ij} \log q_{ij}$$

El primer término no depende de $y_i$, así que solo derivamos el segundo.

Definimos $d_{ij} = \|y_i - y_j\|^2$ y $Z = \sum_{k \neq l}(1 + d_{kl})^{-1}$, entonces $q_{ij} = (1+d_{ij})^{-1}/Z$.

$$\log q_{ij} = -\log(1 + d_{ij}) - \log Z$$

$$\frac{\partial C}{\partial y_i} = -\sum_{j \neq i} p_{ij} \frac{\partial \log q_{ij}}{\partial y_i} + \sum_{j \neq i} \frac{\partial}{\partial y_i}\left[p_{ij} \log p_{ij}\right]$$

Desarrollando y simplificando (después de manejar las derivadas de $Z$ y $d_{ij}$):

$$\frac{\partial C}{\partial y_i} = 4 \sum_{j \neq i} (p_{ij} - q_{ij})(y_i - y_j)(1 + d_{ij})^{-1}$$

### Paso 6: Actualización con momentum

Los puntos se actualizan con descenso de gradiente con momentum:

$$y_i^{(t+1)} = y_i^{(t)} - \eta \frac{\partial C}{\partial y_i} + \alpha(t) \left(y_i^{(t)} - y_i^{(t-1)}\right)$$

donde $\eta$ es la tasa de aprendizaje y $\alpha(t)$ es el momentum (típicamente 0.5 al inicio y 0.8 después).

### Paso 7: Early exaggeration

En las primeras iteraciones (típicamente las primeras 250), multiplicamos todas las $p_{ij}$ por un factor (usualmente 4 o 12). Esto:
- Hace que las fuerzas atractivas sean mucho más fuertes
- Obliga a los clusters a formarse rápidamente
- Deja espacio entre clusters para que después se ajusten finamente

---

## Visualización: Gaussiana vs t-Student

Veamos por qué la distribución t-Student es clave para resolver el problema de hacinamiento.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.linspace(-5, 5, 500)

# Gaussiana vs t-Student
gauss = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)
t_student = 1 / (np.pi * (1 + x**2))  # Cauchy = t con 1 gl

ax = axes[0]
ax.plot(x, gauss, 'b-', linewidth=2.5, label='Gaussiana')
ax.plot(x, t_student, 'r-', linewidth=2.5, label='t-Student (1 gl)')
ax.fill_between(x, gauss, alpha=0.15, color='blue')
ax.fill_between(x, t_student, alpha=0.15, color='red')
ax.set_xlabel('Distancia', fontsize=12)
ax.set_ylabel('Densidad de probabilidad', fontsize=12)
ax.set_title('Comparación de distribuciones', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-5, 5)

# Zoom en las colas
ax = axes[1]
x_tail = np.linspace(1.5, 5, 300)
gauss_tail = np.exp(-x_tail**2 / 2) / np.sqrt(2 * np.pi)
t_tail = 1 / (np.pi * (1 + x_tail**2))

ax.plot(x_tail, gauss_tail, 'b-', linewidth=2.5, label='Gaussiana')
ax.plot(x_tail, t_tail, 'r-', linewidth=2.5, label='t-Student (1 gl)')
ax.fill_between(x_tail, gauss_tail, alpha=0.15, color='blue')
ax.fill_between(x_tail, t_tail, alpha=0.15, color='red')

# Anotaciones
ax.annotate('Colas más pesadas\n→ permite mayor separación\n   entre clusters', 
            xy=(2.5, 0.035), fontsize=10,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='orange'))

ax.set_xlabel('Distancia', fontsize=12)
ax.set_ylabel('Densidad de probabilidad', fontsize=12)
ax.set_title('Zoom en las colas', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("La distribución t-Student asigna más probabilidad a distancias grandes,")
print("permitiendo que los puntos lejanos se separen más en el espacio reducido.")

---

## Ejemplo Numérico Paso a Paso

Trabajemos con 4 puntos en 2D para entender el algoritmo completo. Queremos proyectarlos a 1D (para simplificar la visualización del proceso).

In [ ]:
# 4 puntos en 2D: dos pares cercanos
X_ej = np.array([
    [0.0, 0.0],   # x1
    [1.0, 0.0],   # x2 (cerca de x1)
    [5.0, 0.0],   # x3
    [6.0, 0.0],   # x4 (cerca de x3)
])

n = len(X_ej)
print("="*60)
print("PASO 1: Calcular distancias euclidianas al cuadrado")
print("="*60)

# Matriz de distancias al cuadrado
D = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        D[i,j] = np.sum((X_ej[i] - X_ej[j])**2)

print("\nMatriz de distancias al cuadrado ||xi - xj||²:")
print(f"{'':>6}", end='')
for j in range(n):
    print(f"  x_{j+1:>3}", end='')
print()
for i in range(n):
    print(f"x_{i+1:>3} ", end='')
    for j in range(n):
        print(f"{D[i,j]:>7.1f}", end='')
    print()

In [ ]:
print("="*60)
print("PASO 2: Calcular probabilidades condicionales p(j|i)")
print("="*60)
print("\nUsamos σ = 1.0 para todos los puntos (simplificación)")
print("\nFórmula: p(j|i) = exp(-||xi-xj||²/2σ²) / Σ_k exp(-||xi-xk||²/2σ²)")

sigma = 1.0

# Calcular p(j|i)
P_cond = np.zeros((n, n))
for i in range(n):
    numeradores = np.exp(-D[i] / (2 * sigma**2))
    numeradores[i] = 0  # p(i|i) = 0
    denominador = np.sum(numeradores)
    P_cond[i] = numeradores / denominador

print("\nPara x_1 (σ=1):")
for j in range(n):
    if j != 0:
        num = np.exp(-D[0,j]/(2*sigma**2))
        print(f"  exp(-{D[0,j]:.1f}/2) = exp(-{D[0,j]/2:.1f}) = {num:.6f}")

denom = sum(np.exp(-D[0,j]/(2*sigma**2)) for j in range(1,n))
print(f"  Denominador = {denom:.6f}")
print(f"\n  p(2|1) = {P_cond[0,1]:.6f}  (x2 está cerca → probabilidad ALTA)")
print(f"  p(3|1) = {P_cond[0,2]:.6f}  (x3 está lejos → probabilidad baja)")
print(f"  p(4|1) = {P_cond[0,3]:.6f}  (x4 está más lejos → probabilidad muy baja)")

print("\n\nMatriz completa de P(j|i):")
print(f"{'':>6}", end='')
for j in range(n):
    print(f"{'x_'+str(j+1):>10}", end='')
print()
for i in range(n):
    print(f"x_{i+1:>3} ", end='')
    for j in range(n):
        print(f"{P_cond[i,j]:>10.6f}", end='')
    print()

In [ ]:
print("="*60)
print("PASO 3: Simetrizar → p_ij = (p(j|i) + p(i|j)) / 2n")
print("="*60)

P = (P_cond + P_cond.T) / (2 * n)

print(f"\nn = {n} puntos")
print(f"\nEjemplo: p_12 = (p(2|1) + p(1|2)) / (2·{n})")
print(f"       = ({P_cond[0,1]:.6f} + {P_cond[1,0]:.6f}) / {2*n}")
print(f"       = {P[0,1]:.6f}")

print("\nMatriz simétrica P_ij:")
print(f"{'':>6}", end='')
for j in range(n):
    print(f"{'x_'+str(j+1):>10}", end='')
print()
for i in range(n):
    print(f"x_{i+1:>3} ", end='')
    for j in range(n):
        print(f"{P[i,j]:>10.6f}", end='')
    print()
print(f"\nSuma total de P: {P.sum():.6f} (debe ser ≈ 1.0)")

In [ ]:
print("="*60)
print("PASO 4: Inicializar puntos aleatorios en espacio reducido")
print("="*60)

np.random.seed(42)
Y = np.random.randn(n, 1) * 0.01  # Inicialización pequeña en 1D

print("\nPosiciones iniciales (1D):")
for i in range(n):
    print(f"  y_{i+1} = {Y[i,0]:.6f}")

print("\n" + "="*60)
print("PASO 5: Calcular q_ij con distribución t-Student")
print("="*60)

# Distancias en espacio reducido
D_low = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        D_low[i,j] = np.sum((Y[i] - Y[j])**2)

# q_ij con t-Student
num_q = (1 + D_low)**(-1)
np.fill_diagonal(num_q, 0)
Z = np.sum(num_q)
Q = num_q / Z
Q = np.maximum(Q, 1e-12)

print(f"\nFórmula: q_ij = (1 + ||yi-yj||²)^(-1) / Z")
print(f"Z = Σ (1 + ||yk-yl||²)^(-1) = {Z:.6f}")

print("\nMatriz Q_ij:")
print(f"{'':>6}", end='')
for j in range(n):
    print(f"{'x_'+str(j+1):>10}", end='')
print()
for i in range(n):
    print(f"x_{i+1:>3} ", end='')
    for j in range(n):
        print(f"{Q[i,j]:>10.6f}", end='')
    print()

In [ ]:
print("="*60)
print("PASO 6: Calcular gradiente y actualizar")
print("="*60)

print("\nFórmula: ∂C/∂yi = 4 Σ_j (pij - qij)(yi - yj)(1+||yi-yj||²)^(-1)")

# Calcular gradientes
grad = np.zeros_like(Y)
for i in range(n):
    diff_pq = P[i] - Q[i]  # (p_ij - q_ij)
    diff_y = Y[i] - Y       # (y_i - y_j)
    inv_dist = (1 + D_low[i])**(-1)  # (1+||yi-yj||²)^(-1)
    
    for j in range(n):
        if j != i:
            grad[i] += 4 * diff_pq[j] * diff_y[j] * inv_dist[j]

print("\nGradiente para cada punto:")
for i in range(n):
    print(f"  ∂C/∂y_{i+1} = {grad[i,0]:>10.6f}")
    
    # Desglose
    print(f"    Contribuciones:")
    for j in range(n):
        if j != i:
            contrib = 4 * (P[i,j] - Q[i,j]) * (Y[i,0] - Y[j,0]) * (1 + D_low[i,j])**(-1)
            tipo = "atractiva" if (P[i,j] - Q[i,j]) > 0 else "repulsiva"
            print(f"      j={j+1}: (p-q)={P[i,j]-Q[i,j]:+.6f} → fuerza {tipo}: {contrib:+.6f}")

# Actualizar
lr = 100.0  # Tasa de aprendizaje típica en t-SNE
Y_new = Y - lr * grad

print(f"\nActualización con η = {lr}:")
print(f"y_i(nuevo) = y_i(actual) - η · ∂C/∂yi\n")
for i in range(n):
    print(f"  y_{i+1}: {Y[i,0]:.6f} - {lr}·({grad[i,0]:.6f}) = {Y_new[i,0]:.6f}")

In [ ]:
# Visualizar el proceso de optimización (varias iteraciones)
np.random.seed(42)
Y_opt = np.random.randn(n, 1) * 0.01
Y_prev = Y_opt.copy()
lr = 100.0
momentum = 0.5

# Guardar trayectoria
trayectoria = [Y_opt.copy()]
costs = []

for iteration in range(200):
    # Distancias en espacio reducido
    D_low = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D_low[i,j] = np.sum((Y_opt[i] - Y_opt[j])**2)
    
    # q_ij
    num_q = (1 + D_low)**(-1)
    np.fill_diagonal(num_q, 0)
    Z = np.sum(num_q)
    Q = np.maximum(num_q / Z, 1e-12)
    
    # KL divergence
    mask = P > 0
    cost = np.sum(P[mask] * np.log(P[mask] / Q[mask]))
    costs.append(cost)
    
    # Gradientes
    grad = np.zeros_like(Y_opt)
    for i in range(n):
        for j in range(n):
            if j != i:
                grad[i] += 4 * (P[i,j] - Q[i,j]) * (Y_opt[i] - Y_opt[j]) * (1 + D_low[i,j])**(-1)
    
    # Actualizar con momentum
    Y_new = Y_opt - lr * grad + momentum * (Y_opt - Y_prev)
    Y_prev = Y_opt.copy()
    Y_opt = Y_new
    
    if iteration in [0, 4, 19, 49, 99, 199]:
        trayectoria.append(Y_opt.copy())

# Visualizar
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

# Primera fila: evolución de posiciones
titulos = ['Inicio', 'Iter 5', 'Iter 20', 'Iter 50', 'Iter 100', 'Iter 200']
colores_ej = ['#e74c3c', '#e74c3c', '#3498db', '#3498db']

for idx, (Y_snap, titulo) in enumerate(zip(trayectoria[:4], titulos[:4])):
    ax = axes[0, idx]
    for i in range(n):
        ax.scatter(Y_snap[i,0], 0, c=colores_ej[i], s=150, edgecolors='black', zorder=5)
        ax.annotate(f'$y_{i+1}$', (Y_snap[i,0], 0), textcoords="offset points", 
                   xytext=(0, 15), fontsize=11, ha='center')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_yticks([])
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    ax.grid(True, alpha=0.3, axis='x')

# Posiciones finales
for idx, (Y_snap, titulo) in enumerate(zip(trayectoria[4:], titulos[4:])):
    ax = axes[1, idx]
    for i in range(n):
        ax.scatter(Y_snap[i,0], 0, c=colores_ej[i], s=150, edgecolors='black', zorder=5)
        ax.annotate(f'$y_{i+1}$', (Y_snap[i,0], 0), textcoords="offset points", 
                   xytext=(0, 15), fontsize=11, ha='center')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_yticks([])
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    ax.grid(True, alpha=0.3, axis='x')

# Curva de costo
ax = axes[1, 2]
ax.plot(costs, 'b-', linewidth=2)
ax.set_xlabel('Iteración', fontsize=11)
ax.set_ylabel('KL Divergence', fontsize=11)
ax.set_title('Convergencia del costo', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# Comparación original vs final
ax = axes[1, 3]
ax.scatter(X_ej[:2, 0], [0.3, 0.3], c='#e74c3c', s=100, edgecolors='black', marker='s', label='Original (2D, dim1)')
ax.scatter(X_ej[2:, 0], [0.3, 0.3], c='#3498db', s=100, edgecolors='black', marker='s')
Y_final = trayectoria[-1]
ax.scatter(Y_final[:2, 0], [-0.3, -0.3], c='#e74c3c', s=100, edgecolors='black', marker='o', label='t-SNE (1D)')
ax.scatter(Y_final[2:, 0], [-0.3, -0.3], c='#3498db', s=100, edgecolors='black', marker='o')
ax.set_title('Original vs t-SNE', fontsize=12, fontweight='bold')
ax.set_yticks([-0.3, 0.3])
ax.set_yticklabels(['t-SNE', 'Original'])
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nResultado: t-SNE separó correctamente los dos clusters.")
print(f"Puntos x1,x2 (cluster rojo) quedaron juntos: y1={Y_final[0,0]:.2f}, y2={Y_final[1,0]:.2f}")
print(f"Puntos x3,x4 (cluster azul) quedaron juntos: y3={Y_final[2,0]:.2f}, y4={Y_final[3,0]:.2f}")

---

## Implementación con Código

Ahora implementemos t-SNE desde cero, primero con funciones individuales y luego como una clase completa.

### Funciones

In [ ]:
def distancias_pares(X):
    """
    Calcula la matriz de distancias euclidianas al cuadrado entre todos los pares de puntos.
    
    Usa el truco: ||xi - xj||² = ||xi||² + ||xj||² - 2·xi·xj
    
    Parámetros
    ----------
    X : array de forma (n, d)
        Datos con n puntos en d dimensiones.
    
    Retorna
    -------
    D : array de forma (n, n)
        Matriz de distancias al cuadrado.
    """
    sum_X = np.sum(X**2, axis=1)
    D = sum_X[:, np.newaxis] + sum_X[np.newaxis, :] - 2 * X @ X.T
    np.fill_diagonal(D, 0)
    return np.maximum(D, 0)  # Asegurar no-negatividad por errores numéricos


# Prueba
X_test = np.array([[0, 0], [1, 0], [5, 0], [6, 0]])
D_test = distancias_pares(X_test)
print("Distancias al cuadrado:")
print(D_test)
print(f"\n||x1-x2||² = {D_test[0,1]:.1f} (esperado: 1.0)")
print(f"||x1-x3||² = {D_test[0,2]:.1f} (esperado: 25.0)")

In [ ]:
def perplejidad_fila(distancias_i, sigma):
    """
    Calcula las probabilidades condicionales y la perplejidad para un punto dado.
    
    Parámetros
    ----------
    distancias_i : array de forma (n,)
        Distancias al cuadrado desde el punto i a todos los demás.
    sigma : float
        Ancho de la Gaussiana.
    
    Retorna
    -------
    P_i : array de forma (n,)
        Probabilidades condicionales p(j|i).
    perp : float
        Perplejidad de la distribución.
    """
    # p(j|i) = exp(-d_ij / 2σ²) / Σ exp(-d_ik / 2σ²)
    P_i = np.exp(-distancias_i / (2 * sigma**2))
    sum_P = np.sum(P_i)
    
    if sum_P == 0:
        P_i = np.zeros_like(distancias_i)
        return P_i, 0
    
    P_i = P_i / sum_P
    
    # Entropía: H = -Σ p_j·log2(p_j)
    P_i_pos = P_i[P_i > 1e-12]
    H = -np.sum(P_i_pos * np.log2(P_i_pos))
    
    # Perplejidad = 2^H
    perp = 2**H
    
    return P_i, perp


# Prueba
d_test = np.array([0.0, 1.0, 25.0, 36.0])  # distancias desde x1
d_test_sin_self = np.array([1.0, 25.0, 36.0])  # sin el propio punto
P_test, perp_test = perplejidad_fila(d_test_sin_self, sigma=1.0)
print(f"Probabilidades: {P_test}")
print(f"Perplejidad: {perp_test:.4f}")
print(f"(≈ número efectivo de vecinos)")

In [ ]:
def busqueda_binaria_sigma(distancias_i, perp_objetivo, tol=1e-5, max_iter=50):
    """
    Encuentra el σ que produce la perplejidad deseada usando búsqueda binaria.
    
    Parámetros
    ----------
    distancias_i : array de forma (n-1,)
        Distancias al cuadrado desde el punto i a todos los demás.
    perp_objetivo : float
        Perplejidad deseada.
    tol : float
        Tolerancia para convergencia.
    max_iter : int
        Máximo de iteraciones de búsqueda.
    
    Retorna
    -------
    P_i : array de forma (n-1,)
        Probabilidades condicionales con el σ óptimo.
    sigma : float
        El σ encontrado.
    """
    sigma_min = 1e-10
    sigma_max = 1e4
    sigma = 1.0
    
    for _ in range(max_iter):
        P_i, perp_actual = perplejidad_fila(distancias_i, sigma)
        
        diff = perp_actual - perp_objetivo
        
        if abs(diff) < tol:
            break
        
        if diff > 0:
            # Perplejidad muy alta → σ muy grande → reducir σ
            sigma_max = sigma
        else:
            # Perplejidad muy baja → σ muy pequeño → aumentar σ
            sigma_min = sigma
        
        sigma = (sigma_min + sigma_max) / 2
    
    return P_i, sigma


# Prueba
P_opt, sigma_opt = busqueda_binaria_sigma(d_test_sin_self, perp_objetivo=2.0)
_, perp_verificar = perplejidad_fila(d_test_sin_self, sigma_opt)
print(f"Perplejidad objetivo: 2.0")
print(f"σ encontrado: {sigma_opt:.6f}")
print(f"Perplejidad obtenida: {perp_verificar:.6f}")
print(f"Probabilidades: {P_opt}")

In [ ]:
def calcular_P_conjunta(X, perplejidad=30.0):
    """
    Calcula la matriz de probabilidades conjuntas P para todos los pares de puntos.
    
    Parámetros
    ----------
    X : array de forma (n, d)
        Datos originales.
    perplejidad : float
        Perplejidad deseada (número efectivo de vecinos).
    
    Retorna
    -------
    P : array de forma (n, n)
        Matriz simétrica de probabilidades conjuntas.
    """
    n = X.shape[0]
    D = distancias_pares(X)
    
    P_cond = np.zeros((n, n))
    sigmas = np.zeros(n)
    
    for i in range(n):
        # Extraer distancias sin el propio punto
        mask = np.ones(n, dtype=bool)
        mask[i] = False
        dist_i = D[i, mask]
        
        # Buscar σ óptimo
        P_i, sigma_i = busqueda_binaria_sigma(dist_i, perplejidad)
        sigmas[i] = sigma_i
        
        # Colocar en la fila (respetando el índice del propio punto)
        P_cond[i, mask] = P_i
    
    # Simetrizar: P_ij = (P(j|i) + P(i|j)) / 2n
    P = (P_cond + P_cond.T) / (2 * n)
    P = np.maximum(P, 1e-12)
    
    return P


# Prueba con los 4 puntos
P_test = calcular_P_conjunta(X_test, perplejidad=2.0)
print("Matriz P conjunta (simétrica):")
print(np.round(P_test, 6))
print(f"\nSuma: {P_test.sum():.6f}")
print(f"Simetría verificada: {np.allclose(P_test, P_test.T)}")

In [ ]:
def calcular_Q(Y):
    """
    Calcula la matriz de probabilidades Q en el espacio reducido
    usando la distribución t-Student con 1 grado de libertad.
    
    Parámetros
    ----------
    Y : array de forma (n, d_out)
        Puntos en el espacio reducido.
    
    Retorna
    -------
    Q : array de forma (n, n)
        Matriz de probabilidades.
    num : array de forma (n, n)
        Numeradores (1 + ||yi-yj||²)^(-1), útiles para el gradiente.
    """
    D_low = distancias_pares(Y)
    num = (1 + D_low)**(-1)
    np.fill_diagonal(num, 0)
    Z = np.sum(num)
    Q = num / Z
    Q = np.maximum(Q, 1e-12)
    return Q, num


# Prueba
Y_test = np.array([[0.0], [0.1], [2.0], [2.1]])
Q_test, num_test = calcular_Q(Y_test)
print("Matriz Q (t-Student):")
print(np.round(Q_test, 6))
print(f"\nSuma: {Q_test.sum():.6f}")

In [ ]:
def calcular_gradiente(P, Q, Y, num):
    """
    Calcula el gradiente de la divergencia KL respecto a los puntos Y.
    
    ∂C/∂yi = 4 Σ_j (pij - qij)(yi - yj)(1 + ||yi-yj||²)^(-1)
    
    Parámetros
    ----------
    P : array de forma (n, n)
        Probabilidades del espacio original.
    Q : array de forma (n, n)
        Probabilidades del espacio reducido.
    Y : array de forma (n, d_out)
        Posiciones actuales en espacio reducido.
    num : array de forma (n, n)
        Numeradores (1 + ||yi-yj||²)^(-1).
    
    Retorna
    -------
    grad : array de forma (n, d_out)
        Gradientes para cada punto.
    """
    n, d_out = Y.shape
    grad = np.zeros_like(Y)
    
    # PQ_diff tiene forma (n, n) 
    PQ_diff = P - Q  # (pij - qij)
    
    for i in range(n):
        # (pij - qij) * (1 + ||yi-yj||²)^(-1) para todo j
        weights = PQ_diff[i] * num[i]  # forma (n,)
        # (yi - yj) para todo j
        diff = Y[i] - Y  # forma (n, d_out)
        # Suma ponderada
        grad[i] = 4 * np.sum(weights[:, np.newaxis] * diff, axis=0)
    
    return grad


# Prueba
grad_test = calcular_gradiente(P_test, Q_test, Y_test, num_test)
print("Gradientes:")
for i in range(len(Y_test)):
    print(f"  ∂C/∂y_{i+1} = {grad_test[i,0]:.6f}")

In [ ]:
def kl_divergencia(P, Q):
    """
    Calcula la divergencia KL: KL(P||Q) = Σ pij · log(pij/qij)
    
    Parámetros
    ----------
    P : array de forma (n, n)
    Q : array de forma (n, n)
    
    Retorna
    -------
    kl : float
        Valor de la divergencia KL.
    """
    mask = P > 1e-12
    return np.sum(P[mask] * np.log(P[mask] / Q[mask]))


# Prueba
kl = kl_divergencia(P_test, Q_test)
print(f"KL(P||Q) = {kl:.6f}")

In [ ]:
def tsne(X, n_componentes=2, perplejidad=30.0, n_iter=1000, lr=200.0,
         momentum_inicial=0.5, momentum_final=0.8, exageracion=12.0,
         iter_exageracion=250, semilla=42):
    """
    Implementación completa de t-SNE.
    
    Parámetros
    ----------
    X : array de forma (n, d)
        Datos de entrada.
    n_componentes : int
        Dimensionalidad del espacio reducido (típicamente 2).
    perplejidad : float
        Perplejidad (número efectivo de vecinos, típico 5-50).
    n_iter : int
        Número de iteraciones de optimización.
    lr : float
        Tasa de aprendizaje.
    momentum_inicial : float
        Momentum para las primeras iteraciones.
    momentum_final : float
        Momentum después de la fase de exageración.
    exageracion : float
        Factor de exageración temprana.
    iter_exageracion : int
        Número de iteraciones con exageración.
    semilla : int
        Semilla para reproducibilidad.
    
    Retorna
    -------
    Y : array de forma (n, n_componentes)
        Datos proyectados.
    costos : list
        Historial de la divergencia KL.
    """
    np.random.seed(semilla)
    n = X.shape[0]
    
    # 1. Calcular probabilidades en espacio original
    P = calcular_P_conjunta(X, perplejidad)
    
    # 2. Aplicar exageración temprana
    P_exag = P * exageracion
    
    # 3. Inicialización aleatoria
    Y = np.random.randn(n, n_componentes) * 1e-4
    Y_prev = Y.copy()
    
    costos = []
    
    for t in range(n_iter):
        # Usar P exagerada o normal según la iteración
        P_actual = P_exag if t < iter_exageracion else P
        momentum = momentum_inicial if t < iter_exageracion else momentum_final
        
        # 4. Calcular Q
        Q, num = calcular_Q(Y)
        
        # 5. Calcular costo
        costo = kl_divergencia(P_actual, Q)
        costos.append(costo)
        
        # 6. Calcular gradiente
        grad = calcular_gradiente(P_actual, Q, Y, num)
        
        # 7. Actualizar con momentum
        Y_new = Y - lr * grad + momentum * (Y - Y_prev)
        
        # 8. Centrar (estabilidad numérica)
        Y_new = Y_new - np.mean(Y_new, axis=0)
        
        Y_prev = Y.copy()
        Y = Y_new
    
    return Y, costos


# Prueba con los 6 puntos originales
Y_resultado, costos = tsne(X, n_componentes=2, perplejidad=2.0, n_iter=500, lr=100.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Resultado t-SNE
ax = axes[0]
for i in range(len(X)):
    ax.scatter(Y_resultado[i,0], Y_resultado[i,1], c=colors[i], s=150, edgecolors='black', zorder=5)
    ax.annotate(f'$x_{i+1}$', (Y_resultado[i,0], Y_resultado[i,1]), 
               textcoords="offset points", xytext=(8,8), fontsize=12)
ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
ax.set_title('Resultado t-SNE', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# Convergencia
ax = axes[1]
ax.plot(costos, 'b-', linewidth=2)
ax.set_xlabel('Iteración', fontsize=12)
ax.set_ylabel('KL Divergence', fontsize=12)
ax.set_title('Convergencia', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axvline(x=250, color='red', linestyle='--', alpha=0.5, label='Fin exageración')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

### Clase

In [ ]:
class TSNE:
    """
    t-SNE (t-distributed Stochastic Neighbor Embedding) implementado desde cero.
    
    Reduce la dimensionalidad de los datos preservando la estructura local,
    ideal para visualización de datos de alta dimensión.
    
    Parámetros
    ----------
    n_componentes : int, default=2
        Dimensionalidad del espacio reducido.
    perplejidad : float, default=30.0
        Número efectivo de vecinos. Valores típicos: 5-50.
    n_iter : int, default=1000
        Número de iteraciones de optimización.
    lr : float, default=200.0
        Tasa de aprendizaje.
    momentum_inicial : float, default=0.5
        Momentum durante la fase de exageración.
    momentum_final : float, default=0.8
        Momentum después de la exageración.
    exageracion : float, default=12.0
        Factor de exageración temprana.
    iter_exageracion : int, default=250
        Iteraciones con exageración.
    semilla : int, default=42
        Semilla para reproducibilidad.
    
    Atributos
    ---------
    embedding_ : array de forma (n, n_componentes)
        Datos transformados.
    costos_ : list
        Historial de la divergencia KL.
    P_ : array de forma (n, n)
        Probabilidades conjuntas del espacio original.
    kl_divergencia_ : float
        Divergencia KL final.
    """
    
    def __init__(self, n_componentes=2, perplejidad=30.0, n_iter=1000, lr=200.0,
                 momentum_inicial=0.5, momentum_final=0.8, exageracion=12.0,
                 iter_exageracion=250, semilla=42):
        self.n_componentes = n_componentes
        self.perplejidad = perplejidad
        self.n_iter = n_iter
        self.lr = lr
        self.momentum_inicial = momentum_inicial
        self.momentum_final = momentum_final
        self.exageracion = exageracion
        self.iter_exageracion = iter_exageracion
        self.semilla = semilla
        
        self.embedding_ = None
        self.costos_ = []
        self.P_ = None
        self.kl_divergencia_ = None
    
    def _distancias_pares(self, X):
        """Matriz de distancias euclidianas al cuadrado."""
        sum_X = np.sum(X**2, axis=1)
        D = sum_X[:, np.newaxis] + sum_X[np.newaxis, :] - 2 * X @ X.T
        np.fill_diagonal(D, 0)
        return np.maximum(D, 0)
    
    def _perplejidad_fila(self, dist_i, sigma):
        """Calcula probabilidades condicionales y perplejidad para un punto."""
        P_i = np.exp(-dist_i / (2 * sigma**2))
        sum_P = np.sum(P_i)
        if sum_P == 0:
            return np.zeros_like(dist_i), 0
        P_i = P_i / sum_P
        P_pos = P_i[P_i > 1e-12]
        H = -np.sum(P_pos * np.log2(P_pos))
        return P_i, 2**H
    
    def _busqueda_sigma(self, dist_i, tol=1e-5, max_iter=50):
        """Búsqueda binaria del σ óptimo."""
        sigma_min, sigma_max, sigma = 1e-10, 1e4, 1.0
        for _ in range(max_iter):
            P_i, perp = self._perplejidad_fila(dist_i, sigma)
            diff = perp - self.perplejidad
            if abs(diff) < tol:
                break
            if diff > 0:
                sigma_max = sigma
            else:
                sigma_min = sigma
            sigma = (sigma_min + sigma_max) / 2
        return P_i
    
    def _calcular_P(self, X):
        """Calcula la matriz P conjunta simétrica."""
        n = X.shape[0]
        D = self._distancias_pares(X)
        P_cond = np.zeros((n, n))
        
        for i in range(n):
            mask = np.ones(n, dtype=bool)
            mask[i] = False
            P_cond[i, mask] = self._busqueda_sigma(D[i, mask])
        
        P = (P_cond + P_cond.T) / (2 * n)
        return np.maximum(P, 1e-12)
    
    def _calcular_Q(self, Y):
        """Calcula la matriz Q con distribución t-Student."""
        D = self._distancias_pares(Y)
        num = (1 + D)**(-1)
        np.fill_diagonal(num, 0)
        Q = num / np.sum(num)
        return np.maximum(Q, 1e-12), num
    
    def _gradiente(self, P, Q, Y, num):
        """Calcula el gradiente de la KL divergence."""
        n = Y.shape[0]
        PQ = P - Q
        grad = np.zeros_like(Y)
        for i in range(n):
            w = PQ[i] * num[i]
            grad[i] = 4 * np.sum(w[:, np.newaxis] * (Y[i] - Y), axis=0)
        return grad
    
    def fit_transform(self, X):
        """
        Ajusta el modelo y retorna los datos transformados.
        
        Parámetros
        ----------
        X : array de forma (n, d)
            Datos de entrada.
        
        Retorna
        -------
        Y : array de forma (n, n_componentes)
            Datos en el espacio reducido.
        """
        np.random.seed(self.semilla)
        n = X.shape[0]
        
        # Calcular P
        self.P_ = self._calcular_P(X)
        P_exag = self.P_ * self.exageracion
        
        # Inicialización
        Y = np.random.randn(n, self.n_componentes) * 1e-4
        Y_prev = Y.copy()
        self.costos_ = []
        
        for t in range(self.n_iter):
            P_act = P_exag if t < self.iter_exageracion else self.P_
            mom = self.momentum_inicial if t < self.iter_exageracion else self.momentum_final
            
            Q, num = self._calcular_Q(Y)
            
            # Costo (con P real, no exagerada, para monitoreo justo)
            mask = self.P_ > 1e-12
            costo = np.sum(self.P_[mask] * np.log(self.P_[mask] / Q[mask]))
            self.costos_.append(costo)
            
            grad = self._gradiente(P_act, Q, Y, num)
            
            Y_new = Y - self.lr * grad + mom * (Y - Y_prev)
            Y_new -= np.mean(Y_new, axis=0)
            
            Y_prev = Y.copy()
            Y = Y_new
        
        self.embedding_ = Y
        self.kl_divergencia_ = self.costos_[-1]
        
        return Y
    
    def plot_embedding(self, labels=None, titulo='t-SNE', ax=None, cmap='tab10', s=30):
        """
        Visualiza el embedding 2D.
        
        Parámetros
        ----------
        labels : array, opcional
            Etiquetas para colorear los puntos.
        titulo : str
            Título del gráfico.
        ax : matplotlib axis, opcional
            Eje para dibujar.
        cmap : str
            Mapa de colores.
        s : int
            Tamaño de los puntos.
        """
        if self.embedding_ is None:
            raise ValueError("Primero ejecuta fit_transform()")
        
        show = False
        if ax is None:
            fig, ax = plt.subplots(1, 1, figsize=(8, 6))
            show = True
        
        if labels is not None:
            scatter = ax.scatter(self.embedding_[:, 0], self.embedding_[:, 1],
                               c=labels, cmap=cmap, s=s, alpha=0.7, edgecolors='white', linewidths=0.5)
            plt.colorbar(scatter, ax=ax, label='Clase')
        else:
            ax.scatter(self.embedding_[:, 0], self.embedding_[:, 1],
                      s=s, alpha=0.7, edgecolors='white', linewidths=0.5)
        
        ax.set_xlabel('t-SNE 1', fontsize=12)
        ax.set_ylabel('t-SNE 2', fontsize=12)
        ax.set_title(titulo, fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        if show:
            plt.tight_layout()
            plt.show()
    
    def plot_convergencia(self):
        """
        Visualiza la curva de convergencia de la KL divergence.
        """
        if not self.costos_:
            raise ValueError("Primero ejecuta fit_transform()")
        
        fig, ax = plt.subplots(1, 1, figsize=(8, 4))
        ax.plot(self.costos_, 'b-', linewidth=2)
        ax.axvline(x=self.iter_exageracion, color='red', linestyle='--', 
                   alpha=0.5, label=f'Fin exageración (iter {self.iter_exageracion})')
        ax.set_xlabel('Iteración', fontsize=12)
        ax.set_ylabel('KL Divergence', fontsize=12)
        ax.set_title('Convergencia de t-SNE', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print(f"KL Divergence final: {self.costos_[-1]:.6f}")


print("Clase TSNE definida correctamente ✓")

---

## Experimentos

### 1. Dataset de dígitos (sklearn)

In [ ]:
from sklearn.datasets import load_digits

# Cargar dígitos (imágenes 8x8 → 64 dimensiones)
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Forma de los datos: {X_digits.shape}")
print(f"Clases: {np.unique(y_digits)}")
print(f"Muestras por clase: {[np.sum(y_digits == c) for c in range(10)]}")

# Usar un subconjunto para velocidad (t-SNE desde cero es O(n²))
np.random.seed(42)
idx = np.random.choice(len(X_digits), size=500, replace=False)
X_sub = X_digits[idx]
y_sub = y_digits[idx]

print(f"\nUsando {len(X_sub)} muestras para t-SNE")

# Mostrar algunos dígitos
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for digit in range(10):
    idx_d = np.where(y_sub == digit)[0][0]
    axes[0, digit].imshow(X_sub[idx_d].reshape(8, 8), cmap='gray_r')
    axes[0, digit].set_title(f'{digit}', fontsize=12, fontweight='bold')
    axes[0, digit].axis('off')
    axes[1, digit].axis('off')
    texto = ' '.join([f'{v:.0f}' for v in X_sub[idx_d][:8]])
    axes[1, digit].text(0.5, 0.5, f'[{texto}...]', ha='center', va='center', fontsize=7)

plt.suptitle('Cada dígito es un vector de 64 dimensiones', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Aplicar t-SNE
print("Ejecutando t-SNE (esto toma unos segundos)...")
modelo = TSNE(n_componentes=2, perplejidad=30.0, n_iter=1000, lr=200.0, semilla=42)
Y_digits = modelo.fit_transform(X_sub)
print(f"¡Listo! KL Divergence final: {modelo.kl_divergencia_:.4f}")

modelo.plot_embedding(labels=y_sub, titulo='t-SNE de dígitos manuscritos (64D → 2D)', s=40)

In [ ]:
modelo.plot_convergencia()

### 2. PCA vs t-SNE

Comparemos visualmente cómo cada método proyecta los mismos datos.

In [ ]:
# PCA manual para comparación
X_centered = X_sub - X_sub.mean(axis=0)
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
X_pca_2d = X_centered @ Vt[:2].T

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA
ax = axes[0]
scatter1 = ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y_sub, cmap='tab10', 
                      s=30, alpha=0.7, edgecolors='white', linewidths=0.5)
plt.colorbar(scatter1, ax=ax, label='Dígito')
ax.set_xlabel('PC 1', fontsize=12)
ax.set_ylabel('PC 2', fontsize=12)
ax.set_title('PCA (64D → 2D)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# t-SNE
ax = axes[1]
scatter2 = ax.scatter(Y_digits[:, 0], Y_digits[:, 1], c=y_sub, cmap='tab10',
                      s=30, alpha=0.7, edgecolors='white', linewidths=0.5)
plt.colorbar(scatter2, ax=ax, label='Dígito')
ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
ax.set_title('t-SNE (64D → 2D)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('PCA vs t-SNE en dígitos manuscritos', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Observaciones:")
print("• PCA: los clusters se superponen bastante, difícil distinguir clases")
print("• t-SNE: clusters mucho más separados y compactos")
print("• t-SNE captura la estructura NO LINEAL que PCA no puede ver")

### 3. Efecto de la perplejidad

La perplejidad es el hiperparámetro más importante de t-SNE. Veamos cómo afecta el resultado.

In [ ]:
perplejidades = [5, 15, 30, 50]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, perp in enumerate(perplejidades):
    print(f"Calculando t-SNE con perplejidad={perp}...")
    modelo_p = TSNE(n_componentes=2, perplejidad=perp, n_iter=800, lr=200.0, semilla=42)
    Y_p = modelo_p.fit_transform(X_sub)
    
    ax = axes[idx]
    scatter = ax.scatter(Y_p[:, 0], Y_p[:, 1], c=y_sub, cmap='tab10',
                        s=20, alpha=0.7, edgecolors='white', linewidths=0.3)
    ax.set_title(f'Perplejidad = {perp}', fontsize=13, fontweight='bold')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(True, alpha=0.3)

plt.suptitle('Efecto de la perplejidad en t-SNE', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nInterpretación:")
print("• Perplejidad baja (5): clusters muy apretados, puede fragmentar grupos reales")
print("• Perplejidad media (15-30): buen balance entre estructura local y global")
print("• Perplejidad alta (50): clusters más dispersos, visión más global")
print("\nRegla práctica: perplejidad ≈ √n o entre 5 y 50")

### 4. Dataset Iris — estructura más sutil

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data
y_iris = iris.target
nombres_iris = iris.target_names

print(f"Iris: {X_iris.shape[0]} muestras, {X_iris.shape[1]} dimensiones")
print(f"Clases: {nombres_iris}")

# PCA
X_iris_c = X_iris - X_iris.mean(axis=0)
U_i, S_i, Vt_i = np.linalg.svd(X_iris_c, full_matrices=False)
X_iris_pca = X_iris_c @ Vt_i[:2].T

# t-SNE
print("Ejecutando t-SNE en Iris...")
modelo_iris = TSNE(n_componentes=2, perplejidad=30.0, n_iter=1000, lr=200.0, semilla=42)
Y_iris = modelo_iris.fit_transform(X_iris)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colores_iris = ['#e74c3c', '#2ecc71', '#3498db']

for ax, data, titulo in zip(axes, [X_iris_pca, Y_iris], ['PCA', 't-SNE']):
    for c in range(3):
        mask = y_iris == c
        ax.scatter(data[mask, 0], data[mask, 1], c=colores_iris[c], 
                  label=nombres_iris[c], s=60, alpha=0.7, edgecolors='white', linewidths=0.5)
    ax.set_title(f'{titulo} (4D → 2D)', fontsize=14, fontweight='bold')
    ax.set_xlabel(f'{titulo} 1', fontsize=12)
    ax.set_ylabel(f'{titulo} 2', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

plt.suptitle('Iris: PCA vs t-SNE', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Observación: En Iris (4D), tanto PCA como t-SNE funcionan razonablemente bien")
print("porque la estructura es casi lineal. La ventaja de t-SNE se nota más")
print("en datos de ALTA dimensionalidad con estructuras no lineales.")

### 5. Datos sintéticos no lineales — donde t-SNE brilla

In [ ]:
from sklearn.datasets import make_swiss_roll

# Swiss Roll: una superficie 2D enrollada en 3D
np.random.seed(42)
X_swiss, color_swiss = make_swiss_roll(n_samples=800, noise=0.5)

fig = plt.figure(figsize=(16, 5))

# Original 3D
ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(X_swiss[:, 0], X_swiss[:, 1], X_swiss[:, 2], c=color_swiss, cmap='Spectral', s=10)
ax1.set_title('Swiss Roll (3D)', fontsize=13, fontweight='bold')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# PCA
X_sc = X_swiss - X_swiss.mean(axis=0)
U_s, S_s, Vt_s = np.linalg.svd(X_sc, full_matrices=False)
X_swiss_pca = X_sc @ Vt_s[:2].T

ax2 = fig.add_subplot(132)
ax2.scatter(X_swiss_pca[:, 0], X_swiss_pca[:, 1], c=color_swiss, cmap='Spectral', s=10)
ax2.set_title('PCA (3D → 2D)', fontsize=13, fontweight='bold')
ax2.set_xlabel('PC 1'); ax2.set_ylabel('PC 2')
ax2.grid(True, alpha=0.3)

# t-SNE
print("Ejecutando t-SNE en Swiss Roll...")
modelo_swiss = TSNE(n_componentes=2, perplejidad=30.0, n_iter=1000, lr=200.0, semilla=42)
Y_swiss = modelo_swiss.fit_transform(X_swiss)

ax3 = fig.add_subplot(133)
ax3.scatter(Y_swiss[:, 0], Y_swiss[:, 1], c=color_swiss, cmap='Spectral', s=10)
ax3.set_title('t-SNE (3D → 2D)', fontsize=13, fontweight='bold')
ax3.set_xlabel('t-SNE 1'); ax3.set_ylabel('t-SNE 2')
ax3.grid(True, alpha=0.3)

plt.suptitle('Swiss Roll: estructura no lineal', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nObservaciones:")
print("• PCA 'aplasta' el rollo, mezclando colores distantes")
print("• t-SNE 'desenrolla' la estructura, preservando las vecindades")
print("• Los colores cercanos en el rollo quedan cercanos en t-SNE")

### 6. Visualización del proceso de fuerzas

In [ ]:
# Visualizar cómo evoluciona el embedding durante la optimización
from sklearn.datasets import make_blobs

np.random.seed(42)
X_blobs, y_blobs = make_blobs(n_samples=150, n_features=10, centers=3, 
                               cluster_std=1.5, random_state=42)

# Ejecutar t-SNE guardando snapshots
modelo_anim = TSNE(n_componentes=2, perplejidad=20.0, n_iter=800, lr=200.0, semilla=42)

# Calcular P
P_anim = modelo_anim._calcular_P(X_blobs)
P_exag = P_anim * modelo_anim.exageracion

np.random.seed(42)
n_anim = X_blobs.shape[0]
Y_anim = np.random.randn(n_anim, 2) * 1e-4
Y_prev_anim = Y_anim.copy()

snapshots = []
snap_iters = [0, 10, 30, 75, 250, 500, 800]

for t in range(801):
    if t in snap_iters:
        snapshots.append(Y_anim.copy())
    
    P_act = P_exag if t < 250 else P_anim
    mom = 0.5 if t < 250 else 0.8
    
    Q_a, num_a = modelo_anim._calcular_Q(Y_anim)
    grad_a = modelo_anim._gradiente(P_act, Q_a, Y_anim, num_a)
    
    Y_new_a = Y_anim - 200.0 * grad_a + mom * (Y_anim - Y_prev_anim)
    Y_new_a -= np.mean(Y_new_a, axis=0)
    
    Y_prev_anim = Y_anim.copy()
    Y_anim = Y_new_a

# Visualizar snapshots
fig, axes = plt.subplots(1, len(snap_iters), figsize=(24, 3.5))

colores_blobs = ['#e74c3c', '#2ecc71', '#3498db']

for idx, (snap, it) in enumerate(zip(snapshots, snap_iters)):
    ax = axes[idx]
    for c in range(3):
        mask = y_blobs == c
        ax.scatter(snap[mask, 0], snap[mask, 1], c=colores_blobs[c], 
                  s=15, alpha=0.7, edgecolors='white', linewidths=0.3)
    ax.set_title(f'Iter {it}', fontsize=11, fontweight='bold')
    if it < 250:
        ax.set_title(f'Iter {it}\n(exageración)', fontsize=10, fontweight='bold', color='red')
    ax.set_xticks([]); ax.set_yticks([])
    ax.grid(True, alpha=0.2)

plt.suptitle('Evolución del embedding durante la optimización de t-SNE', 
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("Fases:")
print("1. Inicio (iter 0): puntos aleatorios, sin estructura")
print("2. Exageración (iter 10-250): clusters se forman rápidamente por fuerzas atractivas amplificadas")
print("3. Ajuste fino (iter 250+): posiciones se refinan, clusters se compactan")

---

## Advertencias y buenas prácticas de t-SNE

t-SNE es una herramienta poderosa pero hay que usarla con cuidado:

In [ ]:
# Demostrar que t-SNE no es determinista
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, seed in enumerate([42, 123, 7]):
    modelo_det = TSNE(n_componentes=2, perplejidad=20.0, n_iter=800, lr=200.0, semilla=seed)
    Y_det = modelo_det.fit_transform(X_blobs)
    
    ax = axes[idx]
    for c in range(3):
        mask = y_blobs == c
        ax.scatter(Y_det[mask, 0], Y_det[mask, 1], c=colores_blobs[c],
                  s=25, alpha=0.7, edgecolors='white', linewidths=0.5,
                  label=f'Cluster {c}')
    ax.set_title(f'Semilla = {seed}', fontsize=13, fontweight='bold')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('t-SNE NO es determinista: diferentes semillas → diferentes mapas', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Advertencias importantes:")
print("═" * 60)
print("1. NO interpretar distancias entre clusters")
print("   → Solo las distancias DENTRO de clusters son significativas")
print("")
print("2. NO interpretar tamaños de clusters")
print("   → Un cluster grande en t-SNE no significa más dispersión real")
print("")
print("3. Diferentes ejecuciones dan resultados diferentes")
print("   → Ejecutar varias veces para confirmar patrones")
print("")
print("4. La perplejidad importa mucho")
print("   → Probar varios valores (5, 15, 30, 50)")
print("")
print("5. t-SNE es solo para VISUALIZACIÓN")
print("   → No usar el embedding como features para modelos")

---

## Resumen

| Concepto | Descripción |
|---|---|
| **Objetivo** | Visualizar datos de alta dimensión en 2D preservando estructura local |
| **Espacio original** | Similitudes con distribución Gaussiana ($p_{ij}$) |
| **Espacio reducido** | Similitudes con distribución t-Student ($q_{ij}$) |
| **Función de costo** | Divergencia KL entre P y Q |
| **Optimización** | Descenso de gradiente con momentum |
| **Hiperparámetro clave** | Perplejidad (≈ número de vecinos efectivos) |
| **Early exaggeration** | Amplifica P al inicio para formar clusters rápido |
| **Ventaja vs PCA** | Captura relaciones no lineales |
| **Limitación** | No determinista, solo para visualización, O(n²) |
| **Complejidad** | $O(n^2)$ por iteración (existen aproximaciones $O(n \log n)$) |